# Exploratory data analysis of quality-control events
Portfolio demonstration based on a generic event-log analysis workflow. All records, identifiers, categories, dates and results in this notebook are synthetic. No institutional data, schema or original outputs are included.

The goal is to explore event distributions, temporal coverage and the relationship between observed events and reporting activity. This is an EDA demonstration, not a validated forecasting model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)


## 1. Generate reproducible synthetic data
The relational tables below are fictional and intentionally contain a few missing descriptions and duplicate records to illustrate data quality checks.

In [ ]:
sites = pd.DataFrame({
    'site_id': np.arange(1, 13),
    'site_label': [f'Site {i:02d}' for i in range(1, 13)],
    'group': ['A', 'B', 'C'] * 4
})
categories = pd.DataFrame({
    'category_id': np.arange(1, 7),
    'category_label': [f'Event type {i}' for i in range(1, 7)]
})
locations = pd.DataFrame([(i, j, f'Zone {j}') for i in range(1, 7)
                          for j in range(1, 4)],
                         columns=['category_id', 'zone_id', 'zone_label'])
weeks = pd.date_range('2030-01-07', periods=104, freq='W-MON')
rows = []
for site in sites.site_id:
    start = int(rng.integers(0, 20))
    stop = int(rng.integers(65, 105))
    intensity = rng.uniform(1, 9)
    for week in weeks[start:stop]:
        for _ in range(rng.poisson(intensity)):
            rows.append((site, int(rng.integers(1, 7)), int(rng.integers(1, 4)),
                         week + pd.Timedelta(days=int(rng.integers(0, 7)))))
events = pd.DataFrame(rows, columns=['site_id', 'category_id', 'zone_id', 'timestamp'])
events['event_id'] = np.arange(len(events))
events = pd.concat([events, events.iloc[:5]], ignore_index=True)
sites.loc[0, 'site_label'] = None
print(events.shape)


## 2. Integration and data quality
Validate join cardinality to prevent unintended row multiplication. Diagnose missing values, duplicates and timestamp coverage.

In [ ]:
df = events.merge(sites, on='site_id', how='left', validate='many_to_one')
df = df.merge(categories, on='category_id', how='left', validate='many_to_one')
df = df.merge(locations, on=['category_id', 'zone_id'], how='left', validate='many_to_one')
assert len(df) == len(events)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
print(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Time coverage:', df.timestamp.min(), df.timestamp.max())
df = df.drop_duplicates().dropna(subset=['timestamp']).copy()
for col in ['site_label', 'group', 'category_label', 'zone_label']:
    df[col] = df[col].fillna('Unknown')


## 3. Global exploratory analysis

In [ ]:
top_sites = df.groupby('site_id').size().sort_values(ascending=False)
top_categories = df.groupby('category_label').size().sort_values(ascending=False)
print(top_sites.head())
print(top_categories)
print(top_sites.describe())
fig, ax = plt.subplots()
ax.hist(top_sites, bins=8)
ax.set(xlabel='Recorded events per site', ylabel='Number of sites')
plt.show()


In [ ]:
monthly = df.groupby([df.timestamp.dt.to_period('M'), 'group']).size().unstack(fill_value=0)
monthly.index = monthly.index.to_timestamp()
monthly.plot(figsize=(10, 4), ylabel='Recorded events', title='Monthly events by fictional group')
plt.tight_layout()
plt.show()


In [ ]:
coverage = df.groupby(df.timestamp.dt.to_period('M')).agg(
    events=('event_id', 'size'), reporting_sites=('site_id', 'nunique'))
print(coverage.corr(method='pearson'))
coverage.index = coverage.index.to_timestamp()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(coverage.index, coverage.events, label='Events')
ax.set_ylabel('Events')
ax2 = ax.twinx()
ax2.plot(coverage.index, coverage.reporting_sites, color='tab:orange', label='Sites')
ax2.set_ylabel('Sites with observed events')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
bounds = df.groupby('site_id').timestamp.agg(['min', 'max'])
first = bounds['min'].dt.to_period('M').value_counts().sort_index()
last = bounds['max'].dt.to_period('M').value_counts().sort_index()
pd.concat([first.rename('First observed event'), last.rename('Last observed event')], axis=1).fillna(0).plot.bar(figsize=(10, 4))
plt.ylabel('Sites')
plt.tight_layout()
plt.show()


## 4. Site-level analysis
Inspect weekly counts for the three highest-volume fictional sites. Missing weeks are displayed as zero recorded events, not as verified absence of incidents.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, site in zip(axes, top_sites.head(3).index):
    subset = df[df.site_id == site]
    counts = subset.timestamp.dt.to_period('W').value_counts().sort_index()
    counts = counts.reindex(pd.period_range(counts.index.min(), counts.index.max(), freq='W'), fill_value=0)
    ax.plot(counts.index.to_timestamp(), counts.values)
    ax.set(title=f'Fictional site {site}', ylabel='Events')
plt.tight_layout()
plt.show()


## 5. Interpretation and limitations
- Event counts depend on observation coverage and are not incident rates without an exposure denominator.
- Correlation does not establish causation.
- First and last observed events do not establish actual start or end of operations.
- A zero count may reflect missing reporting rather than zero incidents.
- Before forecasting, verify reporting continuity, data completeness and temporal validation design.

All findings produced here refer exclusively to the generated demonstration data.